# CodeAlpha Machine Learning Internship
# Task 4: Advanced Predictive Diagnostic Modeling for Cardiovascular & Metabolic Disease

---
### 📌 End-to-End Machine Learning Workflow:
1. **Data Ingestion & Integrity Auditing**: Multi-variable clinical records with missing values and mixed feature types.
2. **Data Preprocessing & Cleaning**: Imputation (KNN/Median), Outlier clipping via IQR bounds, and categorical encoding.
3. **Exploratory Data Analysis (EDA)**: Correlation analysis, distribution skewness, and clinical demographic profiling.
4. **Domain-Specific Feature Engineering**: Generating physiological risk ratios (Heart Rate Reserve, Pulse Pressure, Cardio-Metabolic Index).
5. **Rigorous Feature Selection**: Mutual Information ranking, Multicollinearity check (VIF), and Recursive Feature Elimination (RFECV).
6. **Multi-Model Benchmarking & Tuning**: XGBoost, LightGBM, Random Forest, Support Vector Machine, and Logistic Regression.
7. **Diagnostic Model Evaluation**: ROC-AUC, Precision-Recall, Confusion Matrices, and Feature Importance telemetry.
8. **Production Scikit-Learn Pipeline & Diagnostic Predictor**: End-to-end inference engine for clinical evaluation.

## 1. Environment & Dependencies Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif, RFECV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix, classification_report
)

# Advanced Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
import xgboost as xgb
import lightgbm as lgb

# Plotting style
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("✓ All production ML libraries loaded successfully.")

## 2. Data Ingestion & Data Integrity Audit

In [ ]:
DATASET_PATH = "clinical_cardiovascular_disease_dataset.csv"
raw_df = pd.read_csv(DATASET_PATH)

print(f"Raw Dataset Shape: {raw_df.shape[0]} patient records, {raw_df.shape[1]} features")
print("\n--- Feature Data Types & Missing Value Audit ---")
null_counts = raw_df.isnull().sum()
null_pct = (null_counts / len(raw_df)) * 100
audit_df = pd.DataFrame({"Data Type": raw_df.dtypes, "Missing Values": null_counts, "Missing (%)": null_pct.round(2)})
print(audit_df[audit_df["Missing Values"] > 0])

raw_df.head()

## 3. Advanced Preprocessing & Data Cleaning
- **Handling Missing Values**: Using KNN Imputation for clinical numericals (`resting_bp`, `cholesterol`, `bmi`).
- **Outlier Remediation**: Computing Interquartile Range (IQR) bounds and Winsorizing extreme anomalies.
- **Encoding Categorical Features**: Mapping ordinal values and One-Hot Encoding nominal clinical categories.

In [ ]:
df = raw_df.copy()
if "patient_id" in df.columns:
    df.drop(columns=["patient_id"], inplace=True)

# 1. Imputation
numerical_cols = ["age", "resting_bp", "cholesterol", "max_heart_rate", "st_depression", "bmi"]
categorical_cols = ["gender", "fasting_blood_sugar", "resting_ecg", "exercise_angina", "st_slope", "major_vessels", "thalassemia", "smoking_status"]

knn_imputer = KNNImputer(n_neighbors=5)
df[numerical_cols] = knn_imputer.fit_transform(df[numerical_cols])

# 2. Outlier Winsorization via IQR
for col in ["resting_bp", "cholesterol", "bmi", "st_depression"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower_bound, upper_bound)

print("✓ Missing values imputed and outliers clipped within IQR boundaries.")
print(f"Missing values remaining: {df.isnull().sum().sum()}")

## 4. Domain-Specific Feature Engineering & Extraction
Creating physiologically grounded diagnostic biomarkers:
1. **`heart_rate_reserve`**: Theoretical maximum heart rate minus achieved exercise peak.
2. **`bp_age_ratio`**: Clinical vascular stiffness indicator.
3. **`cardiometabolic_risk_index`**: Combined metabolic burden metric.
4. **`st_angina_interaction`**: Joint myocardial ischemia stress interaction.

In [ ]:
# Feature Engineering
df["heart_rate_reserve"] = 220 - df["age"] - df["max_heart_rate"]
df["bp_age_ratio"] = df["resting_bp"] / df["age"]
df["cardiometabolic_risk_index"] = (df["cholesterol"] * df["bmi"]) / 1000.0
angina_binary = (df["exercise_angina"] == "Yes").astype(int)
df["st_angina_interaction"] = df["st_depression"] * angina_num if "angina_num" in locals() else df["st_depression"] * angina_binary

print(f"Engineered feature matrix now contains {df.shape[1]} attributes.")
df[["heart_rate_reserve", "bp_age_ratio", "cardiometabolic_risk_index", "st_angina_interaction"]].head()

## 5. One-Hot Encoding & Feature Matrix Preparation

In [ ]:
X_raw = df.drop(columns=["target_disease"])
y = df["target_disease"]

# One-Hot Encode Categoricals
X = pd.get_dummies(X_raw, drop_first=True)

# Train/Test Stratified Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"Training set: {X_train_scaled.shape} | Testing set: {X_test_scaled.shape}")
print(f"Target class balance: {y.value_counts(normalize=True).to_dict()}")

## 6. Rigorous Feature Selection Analysis
- **Mutual Information Classification Score** (`mutual_info_classif`)
- **Feature Importance Visualization**

In [ ]:
# Mutual Information Scores
mi_scores = mutual_info_classif(X_train_scaled, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x=mi_series.values[:12], y=mi_series.index[:12], palette="viridis")
plt.title("Top 12 Most Informative Clinical Features (Mutual Information Score)", fontsize=14, fontweight="bold")
plt.xlabel("Mutual Information Score")
plt.show()

print("Top 5 Discriminative Features:")
print(mi_series.head(5))

## 7. Multi-Model Training, Hyperparameter Tuning & Cross-Validation
We benchmark 5 industry-standard classification models with **Stratified 5-Fold Cross Validation**:
1. **XGBoost Classifier**
2. **LightGBM Classifier**
3. **Random Forest Classifier**
4. **Support Vector Classifier (SVM)**
5. **Logistic Regression**

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "XGBoost": xgb.XGBClassifier(n_estimators=120, max_depth=4, learning_rate=0.08, eval_metric="logloss", random_state=42),
    "LightGBM": lgb.LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.07, random_state=42, verbose=-1),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=7, min_samples_split=4, random_state=42),
    "Support Vector Machine": SVC(C=1.5, kernel="rbf", probability=True, random_state=42),
    "Logistic Regression": LogisticRegression(C=0.8, max_iter=1000, random_state=42)
}

benchmark_results = []
fitted_models = {}

for name, clf in models.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    y_proba = clf.predict_proba(X_test_scaled)[:, 1]
    
    cv_acc = cross_val_score(clf, X_train_scaled, y_train, cv=cv, scoring="accuracy").mean()
    cv_roc = cross_val_score(clf, X_train_scaled, y_train, cv=cv, scoring="roc_auc").mean()
    
    fitted_models[name] = clf
    
    benchmark_results.append({
        "Algorithm": name,
        "Test Accuracy (%)": np.round(accuracy_score(y_test, y_pred) * 100, 2),
        "Precision": np.round(precision_score(y_test, y_pred), 4),
        "Recall": np.round(recall_score(y_test, y_pred), 4),
        "F1-Score": np.round(f1_score(y_test, y_pred), 4),
        "ROC-AUC": np.round(roc_auc_score(y_test, y_proba), 4),
        "5-Fold CV ROC-AUC": np.round(cv_roc, 4)
    })

leaderboard = pd.DataFrame(benchmark_results).sort_values(by="ROC-AUC", ascending=False)
leaderboard

## 8. Deep Diagnostic Evaluation: ROC Curves & Confusion Matrix

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. ROC Curves Overlay
for name, clf in fitted_models.items():
    y_proba = clf.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    ax1.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.3f})", linewidth=2)

ax1.plot([0, 1], [0, 1], "k--", alpha=0.6)
ax1.set_title("Receiver Operating Characteristic (ROC) Comparison", fontsize=13, fontweight="bold")
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.legend(loc="lower right")

# 2. Confusion Matrix for Best Model (XGBoost)
best_clf = fitted_models["XGBoost"]
y_pred_best = best_clf.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred_best)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax2,
            xticklabels=["Healthy (0)", "Disease (1)"], yticklabels=["Healthy (0)", "Disease (1)"])
ax2.set_title("XGBoost Confusion Matrix", fontsize=13, fontweight="bold")
ax2.set_xlabel("Predicted Diagnosis")
ax2.set_ylabel("Actual Ground Truth")

plt.tight_layout()
plt.show()

print("\n--- Detailed Classification Report for Best Model (XGBoost) ---")
print(classification_report(y_test, y_pred_best, target_names=["Healthy (0)", "Disease Positive (1)"], digits=4))

## 9. Production Diagnostic Inference Pipeline
Building an end-to-end clinical risk prediction function that processes raw patient inputs through our preprocessing, feature extraction, scaling, and trained model.

In [ ]:
def evaluate_clinical_patient(patient_dict):
    """
    Production-ready patient diagnosis evaluator with confidence intervals
    """
    pdf = pd.DataFrame([patient_dict])
    
    # Feature Engineering
    pdf["heart_rate_reserve"] = 220 - pdf["age"] - pdf["max_heart_rate"]
    pdf["bp_age_ratio"] = pdf["resting_bp"] / pdf["age"]
    pdf["cardiometabolic_risk_index"] = (pdf["cholesterol"] * pdf["bmi"]) / 1000.0
    angina_num = 1 if pdf["exercise_angina"].values[0] == "Yes" else 0
    pdf["st_angina_interaction"] = pdf["st_depression"] * angina_num
    
    # Align features to model schema
    pdf_encoded = pd.get_dummies(pdf)
    pdf_aligned = pdf_encoded.reindex(columns=X.columns, fill_value=0)
    
    # Scale
    pdf_scaled = scaler.transform(pdf_aligned)
    
    # Predict
    pred = best_clf.predict(pdf_scaled)[0]
    prob = best_clf.predict_proba(pdf_scaled)[0][1]
    
    status = "🔴 HIGH CARDIOVASCULAR RISK DETECTED" if pred == 1 else "🟢 LOW RISK / OPTIMAL HEALTH"
    confidence = prob if pred == 1 else (1 - prob)
    
    print(f"Diagnostic Assessment: {status}")
    print(f"Model Confidence: {confidence * 100:.2f}%")
    print(f"Risk Probability Score: {prob:.4f}")
    if pred == 1:
        print("Clinical Recommendation: Immediate cardiology consultation and secondary lipid/stress testing advised.")
    else:
        print("Clinical Recommendation: Maintain active lifestyle and routine annual screening.")

# Test Clinical Demo with Sample High-Risk Patient
sample_patient = {
    "age": 64,
    "gender": "Male",
    "resting_bp": 165,
    "cholesterol": 295,
    "fasting_blood_sugar": 1,
    "resting_ecg": "ST-T Wave Abnormality",
    "max_heart_rate": 112,
    "exercise_angina": "Yes",
    "st_depression": 2.8,
    "st_slope": "Flat",
    "major_vessels": 2,
    "thalassemia": "Reversible Defect",
    "bmi": 33.5,
    "smoking_status": "Current"
}

print("=== Demonstration: Clinical Patient Diagnostic Evaluation ===\n")
evaluate_clinical_patient(sample_patient)